# ED LoS @15min — CatBoost Bundle Builder

**Purpose**: Train a CatBoost model for `P(ED LoS > 6h)` using the EXACT 16-feature interface your gate→MLP→SPC pipeline expects, and export a compatible `.joblib` bundle.

**Data**: MIMIC-IV Demo CSVs in `/kaggle/input/mimic-iv-demo-v2-2/` — files used: `edstays.csv`, `triage.csv`, `vitalsign.csv`.

**Snapshot**: Strict time-zero window `t0 .. t0+15min` for vitals; triage is treated as intake snapshot (no triage timestamps in demo).

**Outputs**: Joblib bundle with keys: `pipeline` (CatBoostClassifier), `calibrator` (IsotonicRegression), `features` (16 names), `threshold_watch`, `threshold_urgent`, `label` (`los_gt_6h`), `model_kind` (`catboost`).

In [ ]:

# If CatBoost isn't available, install it
try:
    import catboost  # noqa
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
print("CatBoost ready")


In [ ]:

import os, json, warnings
import numpy as np
import pandas as pd
import duckdb
from datetime import datetime
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from joblib import dump
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

# Data root (MIMIC-IV demo)
DATA_ROOT = "/kaggle/input/mimic-iv-demo-v2-2"

# Exact 16 features from your existing bundle (verified)
FEATURES_EXPECTED = [
  'Tag','t_min','Triage','Leitsymptom','HF','MAP','ICU_Kap','Kap_veraltet',
  't_norm','hat_Labor','Labor_ausstehend','hat_Roentgen','Roentgen_ausstehend',
  'hat_CT','CT_ausstehend','naechste_Aktion'
]

# Label definition: LoS > 6h
LOS_THRESH_MIN = 6 * 60

# Alert budgets
WATCH_RATE  = 0.15
URGENT_RATE = 0.05

print("Config OK")


In [ ]:

# Verify required CSVs exist
req = ["edstays.csv","triage.csv","vitalsign.csv"]
for f in req:
    assert os.path.exists(os.path.join(DATA_ROOT, f)), f"Missing {f} in {DATA_ROOT}"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW edstays   AS SELECT * FROM read_csv_auto('{DATA_ROOT}/edstays.csv',   HEADER=TRUE);")
con.execute(f"CREATE OR REPLACE VIEW triage    AS SELECT * FROM read_csv_auto('{DATA_ROOT}/triage.csv',    HEADER=TRUE);")
con.execute(f"CREATE OR REPLACE VIEW vitalsign AS SELECT * FROM read_csv_auto('{DATA_ROOT}/vitalsign.csv', HEADER=TRUE);")

sql = """

WITH ed AS (
  SELECT
    subject_id,
    hadm_id,
    stay_id,
    CAST(intime  AS TIMESTAMP) AS t0,
    CAST(outtime AS TIMESTAMP) AS outtime
  FROM edstays
  WHERE intime IS NOT NULL AND outtime IS NOT NULL
),

-- TRIAGE: demo has no triage charttime → treat as intake snapshot
triage_clean AS (
  SELECT
    stay_id,
    acuity           AS triage_level,
    chiefcomplaint   AS chief_complaint
  FROM triage
),

-- VITALS: last value within t0..t0+15 using window
vitals_win AS (
  SELECT
    e.stay_id,
    CAST(v.charttime AS TIMESTAMP) AS v_ct,
    v.heartrate,
    v.sbp,
    v.dbp,
    ROW_NUMBER() OVER (
      PARTITION BY e.stay_id
      ORDER BY CAST(v.charttime AS TIMESTAMP) DESC
    ) AS rn
  FROM ed e
  LEFT JOIN vitalsign v
    ON v.stay_id = e.stay_id
   AND CAST(v.charttime AS TIMESTAMP) BETWEEN e.t0 AND e.t0 + INTERVAL 15 MINUTE
),
vitals_15 AS (
  SELECT
    stay_id,
    heartrate AS hf,
    CASE WHEN sbp IS NOT NULL AND dbp IS NOT NULL
         THEN (CAST(sbp AS DOUBLE) + 2.0*CAST(dbp AS DOUBLE)) / 3.0
         ELSE NULL END AS map
  FROM vitals_win
  WHERE rn = 1
),

-- Time features from t0
time_feats AS (
  SELECT
    stay_id,
    EXTRACT(DOW  FROM t0)      AS Tag,     -- 0..6
    15                          AS t_min,  -- fixed snapshot
    EXTRACT(HOUR FROM t0)/24.0  AS t_norm  -- 0..1
  FROM ed
),

-- Assemble 16 features + label
features_16 AS (
  SELECT
    e.subject_id,
    e.hadm_id,
    e.stay_id,

    tf.Tag                                   AS "Tag",
    tf.t_min                                 AS "t_min",
    tr.triage_level                          AS "Triage",
    tr.chief_complaint                       AS "Leitsymptom",
    v15.hf                                   AS "HF",
    v15.map                                  AS "MAP",
    CAST(NULL AS DOUBLE)                     AS "ICU_Kap",          -- not in demo
    CAST(0    AS INTEGER)                    AS "Kap_veraltet",     -- default safe
    tf.t_norm                                AS "t_norm",
    CAST(0    AS INTEGER)                    AS "hat_Labor",        -- not in demo
    CAST(0    AS INTEGER)                    AS "Labor_ausstehend",
    CAST(0    AS INTEGER)                    AS "hat_Roentgen",
    CAST(0    AS INTEGER)                    AS "Roentgen_ausstehend",
    CAST(0    AS INTEGER)                    AS "hat_CT",
    CAST(0    AS INTEGER)                    AS "CT_ausstehend",
    'Reassess'                               AS "naechste_Aktion",

    CAST((EXTRACT(EPOCH FROM (e.outtime - e.t0))/60.0) AS DOUBLE) AS los_min,
    e.t0 AS t0
  FROM ed e
  LEFT JOIN time_feats   tf  ON tf.stay_id = e.stay_id
  LEFT JOIN triage_clean tr  ON tr.stay_id = e.stay_id
  LEFT JOIN vitals_15    v15 ON v15.stay_id = e.stay_id
)

SELECT * FROM features_16;
"""  # noqa

features_df = con.execute(sql).df()
print(features_df.shape)
features_df.head(5)


In [ ]:

df = features_df.copy()
df['los_gt_6h'] = (df['los_min'] >= LOS_THRESH_MIN).astype(int)

missing = [c for c in FEATURES_EXPECTED if c not in df.columns]
assert not missing, f"Missing expected feature columns: {missing}"

# Types
df['Leitsymptom']    = df['Leitsymptom'].astype('category')
df['naechste_Aktion'] = df['naechste_Aktion'].astype('category')
# Triage kept numeric (ordinal). Other columns are numeric/int by construction.

# Drop rows with missing label
df = df.dropna(subset=['los_min']).reset_index(drop=True)

# Temporal + subject-aware split:
# order subjects by earliest t0, take 80% subjects for train, 20% for test
subj_first = df.groupby('subject_id')['t0'].min().sort_values().reset_index()
cut = int(len(subj_first) * 0.8)
train_subj = set(subj_first.iloc[:cut]['subject_id'])
test_subj  = set(subj_first.iloc[cut:]['subject_id'])

train_df = df[df['subject_id'].isin(train_subj)].copy()
test_df  = df[df['subject_id'].isin(test_subj)].copy()

# Within train: most-recent 20% subjects → validation
subj_first_train = train_df.groupby('subject_id')['t0'].min().sort_values().reset_index()
vcut = int(len(subj_first_train) * 0.8)
tr_subj = set(subj_first_train.iloc[:vcut]['subject_id'])
va_subj = set(subj_first_train.iloc[vcut:]['subject_id'])

tr_df = train_df[train_df['subject_id'].isin(tr_subj)].copy()
va_df = train_df[train_df['subject_id'].isin(va_subj)].copy()

for name, part in [('train', tr_df), ('valid', va_df), ('test', test_df)]:
    print(name, part.shape, 'pos_rate=', round(part['los_gt_6h'].mean() if len(part)>0 else float('nan'), 3))


In [ ]:

CAT_FEATURES = [c for c in ['Leitsymptom','naechste_Aktion'] if c in df.columns]

def make_pool(frame: pd.DataFrame):
    X = frame.reindex(columns=FEATURES_EXPECTED)
    y = frame['los_gt_6h'].values
    cat_idx = [X.columns.get_loc(c) for c in CAT_FEATURES]
    return Pool(data=X, label=y, cat_features=cat_idx)

train_pool = make_pool(tr_df)
valid_pool = make_pool(va_df)
test_pool  = make_pool(test_df)

model = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.08,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=3.0,
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

va_pred_raw = model.predict_proba(valid_pool)[:,1]
te_pred_raw = model.predict_proba(test_pool)[:,1] if len(test_df)>0 else np.array([])

if len(va_pred_raw):
    print('AUC valid:', round(roc_auc_score(va_df['los_gt_6h'], va_pred_raw), 3))
if len(te_pred_raw):
    print('AUC test :', round(roc_auc_score(test_df['los_gt_6h'], te_pred_raw), 3))


In [ ]:

# Isotonic calibration on validation
y_va = va_df['los_gt_6h'].values
scores_va = va_pred_raw.astype(float)

if np.unique(scores_va).size < 3:
    calib = None
    va_pred = scores_va
    print("NOTE: few unique scores on validation → skipping calibration.")
else:
    calib = IsotonicRegression(out_of_bounds='clip')
    calib.fit(scores_va, y_va)
    va_pred = calib.transform(scores_va)

# Apply to test if available
if len(te_pred_raw):
    te_pred = calib.transform(te_pred_raw.astype(float)) if calib is not None else te_pred_raw.astype(float)
else:
    te_pred = np.array([])

def threshold_for_rate(scores, rate):
    if len(scores) == 0:
        return 1.0
    k = max(1, int(len(scores) * (1.0 - rate)))
    return float(np.partition(scores, k-1)[k-1])

thr_watch  = threshold_for_rate(va_pred, WATCH_RATE)
thr_urgent = threshold_for_rate(va_pred, URGENT_RATE)

def pr_at_rate(y_true, scores, rate):
    if len(scores)==0:
        return 0.0, 0.0, 1.0
    thr = threshold_for_rate(scores, rate)
    yhat = (scores >= thr).astype(int)
    tp = int(((yhat==1)&(y_true==1)).sum())
    fp = int(((yhat==1)&(y_true==0)).sum())
    fn = int(((yhat==0)&(y_true==1)).sum())
    prec = tp / (tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp / (tp+fn) if (tp+fn)>0 else 0.0
    return prec, rec, thr

def ece(y_true, probs, n_bins=10):
    if len(probs)==0:
        return float('nan')
    bins = np.linspace(0.0, 1.0, n_bins+1)
    idx = np.digitize(probs, bins) - 1
    e = 0.0; cnt = 0
    for b in range(n_bins):
        m = (idx==b)
        if m.sum()==0: 
            continue
        conf = probs[m].mean()
        acc  = y_true[m].mean()
        e += m.sum() * abs(conf-acc)
        cnt += m.sum()
    return float(e / max(cnt,1))

metrics = {}
if len(te_pred):
    metrics.update({
        "AUC_test":   round(roc_auc_score(test_df['los_gt_6h'], te_pred), 3),
        "AUPRC_test": round(average_precision_score(test_df['los_gt_6h'], te_pred), 3),
        "Brier_test": round(brier_score_loss(test_df['los_gt_6h'], te_pred), 3),
        "ECE@10bins_test": round(ece(test_df['los_gt_6h'].values, te_pred, n_bins=10), 3)
    })
    pw, rw, _ = pr_at_rate(test_df['los_gt_6h'].values, te_pred, WATCH_RATE)
    pu, ru, _ = pr_at_rate(test_df['los_gt_6h'].values, te_pred, URGENT_RATE)
    metrics["Watch"]  = {"rate": WATCH_RATE,  "precision": round(pw,3), "recall": round(rw,3), "threshold": round(thr_watch,4)}
    metrics["Urgent"] = {"rate": URGENT_RATE, "precision": round(pu,3), "recall": round(ru,3), "threshold": round(thr_urgent,4)}

print(json.dumps({
    "calibrated": calib is not None,
    "thr_watch": thr_watch, "thr_urgent": thr_urgent,
    **metrics
}, indent=2))


In [ ]:

bundle = {
    "pipeline": model,
    "calibrator": calib,
    "features": FEATURES_EXPECTED,
    "threshold_watch": float(thr_watch),
    "threshold_urgent": float(thr_urgent),
    "label": "los_gt_6h",
    "model_kind": "catboost",
    "created_utc": datetime.utcnow().isoformat() + "Z"
}
out_path = "/kaggle/working/ed_phase2_catboost_bundle.joblib"
dump(bundle, out_path, compress=3)
print("Saved bundle →", out_path)


In [ ]:

def predict_proba_df(df_in: pd.DataFrame, bundle=bundle) -> np.ndarray:
    X = df_in.reindex(columns=bundle["features"], fill_value=0)
    raw = bundle["pipeline"].predict_proba(X)[:,1]
    cal = bundle.get("calibrator")
    if cal is not None:
        raw = np.asarray(cal.transform(raw.ravel()))
    return np.clip(raw, 0, 1)

def predict_label_df(df_in: pd.DataFrame, thr: float=None, bundle=bundle) -> np.ndarray:
    thr = float(bundle.get("threshold_watch") if thr is None else thr)
    return (predict_proba_df(df_in, bundle) >= thr).astype(int)

# Smoke on a few rows (no placeholders)
smoke = features_df.reindex(columns=FEATURES_EXPECTED).fillna(0).head(5)
print("Smoke proba:", predict_proba_df(smoke)[:5])
print("Smoke label:", predict_label_df(smoke)[:5])
